# Filmreviews einlesen und bewerten

**Leitfrage:** Welche Wörter kommen in positiven und negativen Filmreviews vor, und wie können wir mit ihnen eine Review bewerten?

**Arbeitsauftrag:** Schaue dir Abschnitte 1 und 2 an und bearbeite danach die beiden Funktionen in Abschnitt 3 eigenständig. Die Wortlisten sind vorgegeben.

Öffne das Notebook mit einem Python-3-Kernel im Notebookordner. Du brauchst die Grundlagen aus Notebook 01. Die [100 IMDb-Trainingsreviews](data/imdb/README.md) liegen als Textdateien im Repository.

In [1]:
from pathlib import Path
import re

train_dir = Path('data/imdb/train')

## 1. Filmreviews finden und einlesen

### 1.1 Finde die Textdateien

`find_review_paths(path)` erhält einen Ordner. `path.glob('*.txt')` sucht darin nach passenden Namen. `path.rglob('*.txt')` durchsucht zusätzlich die Unterordner und entspricht `path.glob('**/*.txt')`. Wir hängen jede gefundene Datei einzeln an die Liste an und sortieren sie zum Schluss.

Implementiere die Funktion. Sie gibt eine sortierte Liste von Dateipfaden zurück.

[Python-Dokumentation zu `Path.glob()`](https://docs.python.org/3/library/pathlib.html#pathlib.Path.glob) · [`Path.rglob()`](https://docs.python.org/3/library/pathlib.html#pathlib.Path.rglob)

In [6]:
def find_review_paths(path: Path) -> list[Path]:
    return list(path.rglob('*.txt'))
    raise NotImplementedError('Implement find_review_paths')


review_paths = find_review_paths(train_dir)
if not review_paths:
    raise FileNotFoundError(f'No review files found under {train_dir.resolve()}.')

print('Review files:', len(review_paths))
print('Example path:', review_paths[0].relative_to(train_dir))

Review files: 100
Example path: neg/12488_1.txt


### 1.2 Lies eine Datei als String ein

`path.open('r', encoding='utf-8')` öffnet die Datei zum Lesen. `file.read()` liest den gesamten Text als einen String, wenn wir direkt nach dem Öffnen damit beginnen. Beim Verlassen von `with` wird die Datei geschlossen.

Bei großen Dateien kann es sinnvoll sein, zeilenweise zu lesen. `file.readline()` liest die nächste Zeile. Mit `for line in file:` liest du die verbleibenden Zeilen nacheinander.

Implementiere die Funktion gemeinsam. Sie gibt den gesamten Inhalt der Datei als String zurück.

[Python-Dokumentation zu `Path.open()`](https://docs.python.org/3/library/pathlib.html#pathlib.Path.open) · [Methoden von Dateiobjekten](https://docs.python.org/3/tutorial/inputoutput.html#methods-of-file-objects)

In [8]:
def read_review(path: Path) -> str:
    with path.open(encoding='utf8') as f:
        s = f.read()
    return s
    raise NotImplementedError('Implement read_review')


print('Preview:', read_review(review_paths[0])[:200])

positive_reviews = []
negative_reviews = []

for review_path in review_paths:
    review = read_review(review_path)
    if review_path.parent.name == 'pos':
        positive_reviews.append(review)
    elif review_path.parent.name == 'neg':
        negative_reviews.append(review)

print('Positive reviews:', len(positive_reviews))
print('Negative reviews:', len(negative_reviews))

Preview: 0.5/10. This movie has absolutely nothing good about it. The acting is among the worst I have ever seen, what is really amazing is that EVERYONE is awful, not just a few here and there, everyone. The 
Positive reviews: 50
Negative reviews: 50


## 2. Vokabular und häufige Wörter bestimmen

Wir verwenden den einfachen Regex aus Notebook 01. Eine kurze Ersetzung hält `br` aus der Wortzählung heraus. `collect_tokens(texts)` sammelt die Tokens aus einer Liste von Review-Texten. `build_vocabulary(tokens)` gibt die Menge der verschiedenen Tokens zurück.

Implementiert gemeinsam `collect_tokens()`: Ruft für jede Review `tokenize_review()` auf und sammelt die Tokens in einer einzigen Liste. Die Reihenfolge und alle Wiederholungen sollen erhalten bleiben.

In [23]:
def tokenize_review(text: str) -> list[str]:
    text = text.replace('<br />', ' ')
    return re.findall(r'[A-Za-z]+', text)


def collect_tokens(texts: list[str]) -> list[str]:
    result = []
    for text in texts:
        tokens = tokenize_review(text)
        result.extend(tokens)
    return result
    raise NotImplementedError('Implement collect_tokens')


def build_vocabulary(tokens: list[str]) -> set[str]:
    return set(tokens)


positive_tokens = collect_tokens(positive_reviews)
negative_tokens = collect_tokens(negative_reviews)
all_tokens = positive_tokens + negative_tokens
vocabulary = build_vocabulary(all_tokens)

print('Instances (N):', len(all_tokens))
print('Types (|V|):', len(vocabulary))

Instances (N): 12114
Types (|V|): 2799


### Vergleiche die Wortfrequenzen

Für die Häufigkeitsanzeige zählen wir Groß- und Kleinschreibung zusammen. `excluded_words` bestimmt, welche Wörter in der Anzeige fehlen. Mit `set()` zählen wir alle Wörter. Mit `stop_words` lassen wir häufige Wörter wie `the` und `and` weg. `not` bleibt in der Anzeige, weil es eine Bewertung umkehren kann.

Implementiere die Funktion `count_words`. Gehe die Tokens durch, schreibe sie klein und überspringe Wörter aus `excluded_words`. Speichere jedes Wort als Schlüssel in einem Dictionary. Wenn ein Wort zum ersten Mal vorkommt, setze seinen Zähler auf `0`. Erhöhe den Zähler dann mit `+= 1`. Gib das Dictionary zurück, das als `keys` alle Worte enthält und pro Wort die Häufigkeit als `value` angibt.

Die vorgegebene Funktion `most_common_words()` sortiert die Wort-Häufigkeit-Paare und gibt die ersten zehn aus.

[Python-Dokumentation zu `sorted()`](https://docs.python.org/3/library/functions.html#sorted)

In [25]:
stop_words = set((
    'a an and are as at be been but by can could did do does for from had has have he her him his '
    'how i if in into is it its me my of on one or our s she so t than that the their them then '
    'there these they this to up was we were what when where which who will with would you your'
).split())


def count_words(tokens: list[str], excluded_words: set[str]) -> dict[str, int]:
    d: dict[str, int] = {}
    
    for token in tokens:
        word = token.lower()
        if word not in excluded_words:
            if word not in d:
                d[word] = 0
            d[word] += 1
    return d
    raise NotImplementedError('Implement count_words')


def most_common_words(counts: dict[str, int], limit: int = 10) -> list[tuple[str, int]]:
    word_count_pairs = counts.items()
    ranked_pairs = sorted(word_count_pairs, key=lambda pair: pair[1], reverse=True)
    return ranked_pairs[:limit]


print('All words:', most_common_words(count_words(all_tokens, set())))
print('Positive reviews:', most_common_words(count_words(positive_tokens, stop_words)))
print('Negative reviews:', most_common_words(count_words(negative_tokens, stop_words)))

All words: [('the', 675), ('and', 350), ('is', 309), ('a', 303), ('i', 282), ('this', 270), ('of', 250), ('it', 240), ('to', 222), ('movie', 157)]
Positive reviews: [('film', 68), ('movie', 55), ('best', 54), ('great', 53), ('excellent', 34), ('story', 28), ('all', 26), ('very', 26), ('show', 23), ('see', 22)]
Negative reviews: [('movie', 102), ('bad', 72), ('not', 55), ('film', 53), ('acting', 41), ('worst', 41), ('awful', 39), ('even', 37), ('good', 33), ('just', 32)]


## 3. Übung: Bewerte Filmreviews mit Wortlisten

### 3.1 Zähle die Treffer und berechne den Score

`score_review()` erhält den Rohtext und zwei Mengen mit positiven und negativen Wörtern. Rufe `tokenize_review(text)` auf und gehe die Tokens nacheinander durch. Zähle ein Token als Treffer in der passenden Menge, wenn es dort genau so geschrieben steht. Jedes Vorkommen zählt, auch bei Wiederholungen.

Gib drei Werte in dieser Reihenfolge zurück: positive Treffer, negative Treffer und den Score (positive Treffer minus negative Treffer). Für `great great bad` ist das Ergebnis `(2, 1, 1)`.

In [14]:
positive_words = {
    'amazing', 'beautiful', 'best', 'brilliant', 'charming', 'enjoy', 'enjoyable', 'excellent',
    'fantastic', 'funny', 'good', 'great', 'impressive', 'like', 'love', 'moving', 'perfect',
    'recommend', 'superb', 'wonderful',
}
negative_words = {
    'annoying', 'awful', 'bad', 'bland', 'boring', 'cheap', 'disappointing', 'dull', 'hate',
    'horrible', 'mediocre', 'pointless', 'poor', 'ridiculous', 'silly', 'stupid', 'terrible',
    'ugly', 'unwatchable', 'worst',
}

In [18]:
def score_review(text: str, positive_words: set[str], negative_words: set[str]) -> tuple[int, int, int]:
    tokens = tokenize_review(text)
    positive, negative = 0, 0
    for token in tokens:
        if token in positive_words:
            positive += 1
        elif token in negative_words:
            negative += 1
    return positive, negative, positive - negative
    raise NotImplementedError('Implement score_review')

### 3.2 Ordne den Score einer Klasse zu

`predict_sentiment(score)` erhält nur den Score. Gib `'pos'` bei einem Score größer als null, `'neg'` bei einem Score kleiner als null und `'undecided'` bei null zurück. Null kann bedeuten, dass beide Trefferzahlen gleich sind oder dass es überhaupt keine Treffer gab.

In [19]:
def predict_sentiment(score: int) -> str:
    if score > 0:
        return 'pos'
    if score < 0:
        return 'neg'
    return 'undecided'
    
    raise NotImplementedError('Implement predict_sentiment')

### 3.3 Schau dir die Ergebnisse an

Führe die Auswertung aus und vergleiche die Zuordnung mit der Klasse im Ordnernamen. Mit `review_paths` statt `example_paths` kannst du alle 100 Reviews ausgeben.

Vergleiche den Score von `good` und `not good`.

In [20]:
def show_scores(paths: list[Path], positive_words: set[str], negative_words: set[str]) -> None:
    print(f'{"File":<15} {"Label":<6} {"Pos":>4} {"Neg":>4} {"Score":>6}  Prediction')
    for path in paths:
        positive_hits, negative_hits, score = score_review(
            read_review(path), positive_words, negative_words
        )
        prediction = predict_sentiment(score)
        print(f'{path.name:<15} {path.parent.name:<6} {positive_hits:>4} '
              f'{negative_hits:>4} {score:>6}  {prediction}')


example_paths = [
    train_dir / 'pos' / '10567_9.txt',
    train_dir / 'neg' / '12366_1.txt',
    train_dir / 'pos' / '12275_9.txt',
    train_dir / 'pos' / '10589_10.txt',
    train_dir / 'neg' / '11886_1.txt',
]

show_scores(example_paths, positive_words, negative_words)
print('Score for good:', score_review('good', positive_words, negative_words)[2])
print('Score for not good:', score_review('not good', positive_words, negative_words)[2])

File            Label   Pos  Neg  Score  Prediction
10567_9.txt     pos       3    0      3  pos
12366_1.txt     neg       1    7     -6  neg
12275_9.txt     pos       0    0      0  undecided
10589_10.txt    pos       0    0      0  undecided
11886_1.txt     neg       1    1      0  undecided
Score for good: 1
Score for not good: 1


### Sind unentschiedene Reviews wirklich neutral?

Öffne die `.txt`-Dateien zu den Reviews mit der Vorhersage `undecided`. Nutze dafür den Dateinamen und den Ordner `pos` oder `neg` aus der Tabelle.

Wie würdest du die Reviews nach dem Lesen bewerten? Nenne eine konkrete Textstelle, die deine Einschätzung stützt. Untersuche anhand der Trefferzahlen und der Wortlisten, warum die Regel trotzdem einen Score von `0` liefert.

Untersuche dafür die Longstreet-Review [10589_10.txt](data/imdb/train/pos/10589_10.txt).

10589_10 soll als positive bewertet werden. Es gibt aber keine Worte, die zu positiven Wortliste passen.